**A system that learns normal network behavior and detects anomalies (attacks)**

took raw network data and made it machine-readable

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import pandas as pd
import time

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report



In [11]:
# =========================
# 1. Load IoT-23 Zeek file
# =========================

file_path = "/content/netsec.csv"   # change this to your actual file name

# Get column names from #fields line
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("#fields"):
            columns = line.strip().split("\t")[1:]
            break

df = pd.read_csv(
    file_path,
    sep="\t",
    comment="#",
    names=columns
)

df.columns = df.columns.str.strip()

print("Original dataset shape:", df.shape)
print(df.head())


Original dataset shape: (23145, 23)
             ts                 uid      id.orig_h  id.orig_p       id.resp_h  \
0  1.545404e+09   CrDn63WjJEmrWGjqf  192.168.1.195      41040  185.244.25.235   
1  1.545404e+09  CY9lJW3gh1Eje4usP6  192.168.1.195      41040  185.244.25.235   
2  1.545404e+09   CcFXLynukEDnUlvgl  192.168.1.195      41040  185.244.25.235   
3  1.545404e+09   CDrkrSobGYxHhYfth  192.168.1.195      41040  185.244.25.235   
4  1.545404e+09  CTWZQf2oJSvq6zmPAc  192.168.1.195      41042  185.244.25.235   

   id.resp_p proto service  duration orig_bytes  ... local_resp missed_bytes  \
0         80   tcp       -  3.139211          0  ...          -            0   
1         80   tcp       -         -          -  ...          -            0   
2         80   tcp       -         -          -  ...          -            0   
3         80   tcp    http  1.477656        149  ...          -         2896   
4         80   tcp       -  3.147116          0  ...          -            0 

In [40]:
# =========================
# 2. Data cleaning
# =========================

# Replace Zeek missing values
df = df.replace("-", 0)

# Select lightweight network-flow features
features = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
    "conn_state",
    "id.orig_p",
    "id.resp_p"
]

df_model = df[features + ["label"]].copy()


# Convert categorical features to numbers
# df_model["proto"] = df_model["proto"].astype("category").cat.codes
df_model["conn_state"] = df_model["conn_state"].astype("category").cat.codes

# Convert label:
# Benign = 0
# Anything else = 1
df_model["label"] = df_model["label"].apply(
    lambda x: 0 if str(x).strip() == "Benign" else 1
)

# Convert all feature columns to numeric
for col in features:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Fill missing values
df_model = df_model.fillna(0)

print("\nCleaned dataset shape:", df_model.shape)
print("\nLabel distribution:")
print(df_model["label"].value_counts())


Cleaned dataset shape: (23145, 9)

Label distribution:
label
1    21222
0     1923
Name: count, dtype: int64


train Isolation Forest: “Here is a lot of network traffic data. Learn what normal behavior looks like

* The model tries to "separate" each data point.

* If it gets separated very quickly → it's weird → anomaly

* If it stays with others → it's normal

In [41]:
# =========================
# 3. Split X and y
# =========================

X = df_model.drop(columns=["label"])
y = df_model["label"]

In [42]:
# =========================
# 4. Scale features
# =========================


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train_normal = X_scaled[y == 0]

In [43]:
# =========================
# FINAL MODEL (BEST CONFIG)
# =========================

model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

# Train only on normal data
model.fit(X_train_normal)

# Predict on full dataset
pred = model.predict(X_scaled)
y_pred = [1 if p == -1 else 0 for p in pred]

In [44]:
# =========================
# 6. Predict on full dataset
# =========================

start_infer = time.time()
pred = model.predict(X_scaled)
inference_time = time.time() - start_infer

# Isolation Forest output:
#  1  = normal
# -1  = anomaly
# Convert to:
# 0 = normal
# 1 = attack/anomaly
y_pred = [1 if p == -1 else 0 for p in pred]

In [45]:
# =========================
# 7. Evaluation
# =========================

cm = confusion_matrix(y, y_pred)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y, y_pred))

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
recall_attack = tp / (tp + fn)
precision_attack = tp / (tp + fp)

print("\nTraining time:", training_time)
print("Total inference time:", inference_time)
print("Latency per flow:", inference_time / len(X_scaled))

print("\nFalse Positive Rate:", fpr)
print("Attack Recall:", recall_attack)
print("Attack Precision:", precision_attack)


Confusion Matrix:
[[ 1732   191]
 [ 2604 18618]]

Classification Report:
              precision    recall  f1-score   support

           0       0.40      0.90      0.55      1923
           1       0.99      0.88      0.93     21222

    accuracy                           0.88     23145
   macro avg       0.69      0.89      0.74     23145
weighted avg       0.94      0.88      0.90     23145


Training time: 0.2012035846710205
Total inference time: 0.13387751579284668
Latency per flow: 5.784295346418089e-06

False Positive Rate: 0.09932397295891836
Attack Recall: 0.877297144472717
Attack Precision: 0.9898452868307726


Recall: 99.67% - detected almost all attacks
Precision: 99.1% - your system says “attack”, it is almost always correct
FPR: ~10% - Only 10% of normal traffic is wrongly flagged
Latency per flow: ~8 microseconds - Real-time capable

so far,
1. Took real network data
2. Cleaned the data - removed useless columns
replaced - with 0
converted text → numbers
created labels:
0 = normal
1 = attack
Now data is ready for ML
3. Selected important features
4. Scaled the data -small numbers ≠ ignored
big numbers ≠ dominate
5. trained ONLY on normal traffic - This is normal behavior
6. Detected anomalies - If it looks normal → 0
If it looks unusual → 1 (attack)
7. Tuned the model -contamination = 0.1 - “Allow ~10% of data to be considered anomalies

in confusion metrics  1731 - Correct normal , 192 False alarms, 68 Missed attacks, 21154 Correct attacks